# 📊 Lead-Lag Playground

`panel_long_dataset.xlsx` 하나와 `playground.py` 하나면 모든 분석이 돌아갑니다.

- **데이터**: `panel_long_dataset.xlsx` 의 `panel_long` 시트 (롱포맷: 기업·분기·지표·값)
- **엔진**: `playground.py` (로드/조회 · 변환 · 시차상관 통계 · 시각화 함수들)

아래 스텝을 위에서부터 실행하세요. **입력값(기업·지표·변환)을 바꿔가며** 탐색하면 됩니다.

### 핵심 개념
- **lead-lag(선후행)**: X가 Y보다 k분기 *앞서* 움직이면 “X가 Y를 k분기 **선행**”.
- **시차상관**: X(t)와 Y(t+k)의 상관을 여러 k에 대해 계산 → 가장 강한 k가 추정 시차.
- 지표(item)에는 원천(revenue, inventory, accounts_payable…)과 **파생(DIO, DSO, CCC, OCF_MARGIN, REVENUE_GROWTH_YOY…)** 이 이미 다 들어있어 이름으로 바로 조회합니다.

## Step 1 — 데이터 로드

In [ ]:
%matplotlib inline
import pandas as pd
from IPython.display import display
import playground as pg          # 엔진 (같은 폴더의 playground.py)

long = pg.load_panel()           # panel_long_dataset.xlsx 로드
print('행:', len(long), '| 기업:', long.ticker.nunique(), '| 지표:', long.item.nunique(),
      '| 기간:', long.quarter.min(), '~', long.quarter.max())
display(long.head(4))

## Step 2 — 무엇을 분석할 수 있나 (지표·기업 목록)

`list_items` 로 지표 이름(그대로 아래 스텝에 넣으면 됨)과 커버리지를, `list_companies` 로 기업 목록을 확인합니다.

In [ ]:
print('■ 지표(item) — 이름을 그대로 x_item / y_item 에 사용')
display(pg.list_items(long))

print('■ 기업(ticker) — 예: AI Chip 섹터')
display(pg.list_companies(long, section='ai_chip'))

## Step 3 — 시계열 훑어보기

여러 (기업, 지표)를 겹쳐 그려 흐름을 눈으로 확인합니다. `tf`(변환)를 바꿔보세요: `level / qoq / yoy / rolling_yoy / zscore` 등.

In [ ]:
pg.plot_series(long, [('NVDA','revenue'), ('MU','revenue'), ('AVGO','revenue')], tf='yoy');

## Step 4 — 두 변수 Lead-Lag 검정 (핵심)

`lead_lag(long, X기업, X지표, Y기업, Y지표, x_tf, y_tf, expected)` — X가 Y를 몇 분기 선행하는지 통계 검정.
- **x_tf / y_tf**: 변환 (`level/qoq/yoy/rolling_yoy/change_qoq/zscore`)
- **expected**: 기대 부호 `'+'` 또는 `'-'` (예: 매출→매출 `+`, CCC→FCF마진 `-`)
- 반환: `best_lag`(선행 분기), `pearson`, `spearman`, `p_value`, `n`, `significant`

In [ ]:
# ============ 여기만 바꿔가며 실험 ============
X_TICKER, X_ITEM, X_TF = 'NVDA', 'revenue', 'yoy'     # 선행 후보
Y_TICKER, Y_ITEM, Y_TF = 'MU',   'revenue', 'yoy'     # 후행 후보
EXPECTED = '+'                                          # 기대 부호
# =============================================

res = pg.lead_lag(long, X_TICKER, X_ITEM, Y_TICKER, Y_ITEM, x_tf=X_TF, y_tf=Y_TF, expected=EXPECTED)
print(res['verdict'])
display(pd.Series({k: res[k] for k in ['best_lag','pearson','spearman','p_value','n','significant']}))
print('전체 시차상관 표:')
display(res['table'].round(3))

## Step 5 — 시각화 (시계열 + 시차상관 2패널)

In [ ]:
pg.plot_pair(long, X_TICKER, X_ITEM, Y_TICKER, Y_ITEM, x_tf=X_TF, y_tf=Y_TF, expected=EXPECTED);

## Step 6 — 여러 조합 한 번에 스캔

관심 있는 (X→Y) 조합 목록을 돌려 결과 표로 비교합니다. 예: 여러 회사의 AP → 특정 공급사 매출.

In [ ]:
# (X기업, X지표, Y기업, Y지표) 목록
combos = [
    ('MSFT','accounts_payable','NVDA','revenue'),
    ('NVDA','revenue','MU','revenue'),
    ('NVDA','revenue','AMKR','revenue'),
    ('AMD','revenue','WDC','revenue'),
]
rows = []
for xt, xi, yt, yi in combos:
    r = pg.lead_lag(long, xt, xi, yt, yi, x_tf='yoy', y_tf='yoy')
    if r['status'] == 'ok':
        rows.append({'X': f'{xt}:{xi}', 'Y': f'{yt}:{yi}', 'lag': r['best_lag'],
                     'r': r['pearson'], 'ρ': r['spearman'], 'p': r['p_value'],
                     'n': r['n'], '유의': r['significant']})
display(pd.DataFrame(rows))